# Comprehensive Behavioral Economics: A Journey Through Human Decision-Making

*"The first principle is that you must not fool yourself — and you are the easiest person to fool."* — Richard Feynman

Welcome to our exploration of behavioral economics, where we'll uncover the fascinating ways humans actually make decisions versus how traditional economic theory assumes they do. I'm Dr. Ian Helfrich, and today we're going to take a journey that combines rigorous mathematical analysis with intuitive understanding — much like Feynman would approach physics, but applied to the beautiful complexity of human economic behavior.


Imagine you're offered two choices:
- **Option A**: A guaranteed $50
- **Option B**: A 50% chance of winning $100, 50% chance of winning nothing

Traditional economic theory says you should be indifferent — both have the same expected value of $50. But most people aren't indifferent. They have preferences that seem to violate the basic axioms of rational choice theory. This isn't because people are "irrational" — it's because human decision-making follows different, but systematic, patterns.

Today, we'll explore these patterns through five fundamental concepts:
1. **Prospect Theory** — How we evaluate gains and losses
2. **Loss Aversion** — Why losses hurt more than equivalent gains feel good
3. **Mental Accounting** — How we categorize and treat money differently
4. **Anchoring & Adjustment** — How initial information shapes our judgments
5. **Framing Effects** — How the presentation of information affects our choices

Each section will build from intuitive examples to rigorous mathematical formulations, complete with interactive demonstrations and agent-based simulations that show these principles in action.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown
import seaborn as sns
from scipy import optimize
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

import sys
sys.path.append('../utils')
sys.path.append('../simulations')

from economic_functions import *
from prospect_theory_agents import *
from market_simulation import *

## 1. Prospect Theory: The Foundation of Behavioral Economics

*"It doesn't matter how beautiful your theory is, it doesn't matter how smart you are. If it doesn't agree with experiment, it's wrong."* — Richard Feynman

Prospect Theory, developed by Daniel Kahneman and Amos Tversky in 1979, revolutionized our understanding of decision-making under risk. It emerged from careful observation of how people actually behave, not how they "should" behave according to expected utility theory.


Traditional expected utility theory assumes people evaluate outcomes in absolute terms. But Kahneman and Tversky discovered something profound: **people evaluate outcomes relative to a reference point**, and they treat gains and losses very differently.



The value function $v(x)$ captures how people perceive gains and losses relative to a reference point:

$$v(x) = \begin{cases}
x^\alpha & \text{if } x \geq 0 \text{ (gains)} \\
-\lambda(-x)^\beta & \text{if } x < 0 \text{ (losses)}
\end{cases}$$

Where:
- $\alpha, \beta \in (0,1)$ capture **diminishing sensitivity**
- $\lambda > 1$ captures **loss aversion**
- $x$ is the outcome relative to the reference point


$$w(p) = \frac{p^\gamma}{(p^\gamma + (1-p)^\gamma)^{1/\gamma}}$$


$$V = \sum_{i=1}^n w(p_i) \cdot v(x_i)$$

In [ ]:
def interactive_prospect_theory():
    @widgets.interact(
        alpha=widgets.FloatSlider(value=0.88, min=0.1, max=1.0, step=0.01, description='α (gain curvature)'),
        beta=widgets.FloatSlider(value=0.88, min=0.1, max=1.0, step=0.01, description='β (loss curvature)'),
        lambda_param=widgets.FloatSlider(value=2.25, min=1.0, max=5.0, step=0.05, description='λ (loss aversion)'),
        gamma=widgets.FloatSlider(value=0.61, min=0.1, max=1.0, step=0.01, description='γ (prob. weighting)')
    )
    def plot_prospect_theory(alpha, beta, lambda_param, gamma):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        x = np.linspace(-100, 100, 1000)
        v = prospect_value_function(x, alpha, beta, lambda_param)
        
        ax1.plot(x, v, 'b-', linewidth=2, label='Prospect Theory Value Function')
        ax1.axhline(y=0, color='k', linestyle='--', alpha=0.3)
        ax1.axvline(x=0, color='k', linestyle='--', alpha=0.3)
        ax1.set_xlabel('Outcome (relative to reference point)')
        ax1.set_ylabel('Subjective Value')
        ax1.set_title('Value Function')
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        p = np.linspace(0.01, 0.99, 100)
        w = probability_weighting_function(p, gamma)
        
        ax2.plot(p, w, 'r-', linewidth=2, label='Probability Weighting Function')
        ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Linear (Rational)')
        ax2.set_xlabel('Objective Probability')
        ax2.set_ylabel('Decision Weight')
        ax2.set_title('Probability Weighting Function')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        plt.show()
        
        print(f"Current parameters:")
        print(f"α = {alpha:.2f}, β = {beta:.2f}, λ = {lambda_param:.2f}, γ = {gamma:.2f}")
        
        if lambda_param > 1:
            print(f"\n💡 Loss aversion: A loss of $10 feels like a gain of ${10*lambda_param:.1f}")
        
        if gamma < 1:
            print(f"💡 Probability distortion: 10% chance feels like {probability_weighting_function(0.1, gamma)*100:.1f}% chance")

interactive_prospect_theory()

## 2. Loss Aversion: Why Losses Loom Larger

*"The pain of losing is psychologically twice as powerful as the pleasure of gaining."* — Daniel Kahneman

Loss aversion is perhaps the most robust finding in behavioral economics. It's not just that people dislike losses — it's that they dislike them disproportionately compared to equivalent gains.


One of the most striking demonstrations of loss aversion is the endowment effect. Once we own something, we value it more highly than we did before we owned it.


We can model the endowment effect using prospect theory:

$$\frac{WTA}{WTP} = 1 + \frac{1}{\lambda}$$

Where WTA is willingness to accept and WTP is willingness to pay.

In [ ]:
def endowment_effect_simulation():
    @widgets.interact(
        lambda_param=widgets.FloatSlider(value=2.25, min=1.0, max=5.0, step=0.1, description='Loss Aversion (λ)'),
        n_agents=widgets.IntSlider(value=100, min=50, max=200, step=10, description='Number of Agents'),
        item_value=widgets.FloatSlider(value=10, min=5, max=20, step=1, description='True Item Value ($)')
    )
    def simulate_trading(lambda_param, n_agents, item_value):
        np.random.seed(42)
        
        owners = np.random.choice([True, False], size=n_agents)
        
        wta_values = []
        wtp_values = []
        
        for i in range(n_agents):
            noise = np.random.normal(0, 0.5)
            true_value = item_value + noise
            
            if owners[i]:
                wta = true_value * (1 + 1/lambda_param)
                wta_values.append(wta)
            else:
                wtp = true_value
                wtp_values.append(wtp)
        
        trades = 0
        for wta in wta_values:
            for wtp in wtp_values:
                if wtp >= wta:
                    trades += 1
                    break
        
        trade_rate = trades / len(wta_values) * 100
        actual_ratio = np.mean(wta_values) / np.mean(wtp_values)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        ax1.hist(wta_values, alpha=0.7, label=f'WTA (Owners)', bins=20, color='red')
        ax1.hist(wtp_values, alpha=0.7, label=f'WTP (Non-owners)', bins=20, color='blue')
        ax1.set_xlabel('Value ($)')
        ax1.set_ylabel('Frequency')
        ax1.set_title('Distribution of WTA vs WTP')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        lambda_range = np.linspace(1, 5, 50)
        predicted_ratios = 1 + 1/lambda_range
        
        ax2.plot(lambda_range, predicted_ratios, 'g-', linewidth=2, label='Theoretical WTA/WTP Ratio')
        ax2.axhline(actual_ratio, color='red', linestyle='--', label=f'Observed Ratio: {actual_ratio:.2f}')
        ax2.axvline(lambda_param, color='blue', linestyle=':', label=f'Current λ: {lambda_param:.2f}')
        ax2.set_xlabel('Loss Aversion Parameter (λ)')
        ax2.set_ylabel('WTA/WTP Ratio')
        ax2.set_title('Endowment Effect Magnitude')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"📊 Simulation Results:")
        print(f"• {len(wta_values)} owners, {len(wtp_values)} non-owners")
        print(f"• Mean WTA: ${np.mean(wta_values):.2f}")
        print(f"• Mean WTP: ${np.mean(wtp_values):.2f}")
        print(f"• WTA/WTP Ratio: {actual_ratio:.2f}")
        print(f"• Trade Rate: {trade_rate:.1f}% (vs 50% predicted by standard theory)")
        print(f"\n💡 With λ = {lambda_param:.2f}, owners demand {((actual_ratio-1)*100):.0f}% more than non-owners will pay!")

endowment_effect_simulation()

## 3. Mental Accounting: The Psychology of Money

*"Money is fungible, but the mind is not."* — Richard Thaler

Traditional economics assumes money is fungible — a dollar is a dollar, regardless of its source or intended use. But behavioral economics reveals that people treat money very differently depending on how they categorize it mentally.


We can model mental accounting using a modified utility function:

$$U = \sum_{i=1}^n w_i \cdot u_i(c_i)$$

Where $c_i$ is consumption in category $i$, $u_i(\cdot)$ is the utility function for category $i$, and $w_i$ is the weight/importance of category $i$.

In [ ]:
def mental_accounting_demo():
    @widgets.interact(
        entertainment_weight=widgets.FloatSlider(value=1.5, min=0.5, max=3.0, step=0.1, description='Entertainment Weight'),
        general_weight=widgets.FloatSlider(value=1.0, min=0.5, max=3.0, step=0.1, description='General Weight'),
        percentage_bias=widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description='Percentage Bias')
    )
    def analyze_mental_accounting(entertainment_weight, general_weight, percentage_bias):
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        theater_costs = [100 * entertainment_weight, 50 * general_weight]
        theater_names = ['Lost Ticket', 'Lost Cash']
        
        bars1 = ax1.bar(theater_names, theater_costs, color=['red', 'blue'], alpha=0.7)
        ax1.axhline(50, color='green', linestyle='--', label='Actual Cost ($50)')
        ax1.set_ylabel('Perceived Cost ($)')
        ax1.set_title('Theater Ticket Mental Accounting')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        cheap_item_pct = 10/50 * 100
        expensive_item_pct = 10/1000 * 100
        
        perceived_values = [
            cheap_item_pct * percentage_bias,
            expensive_item_pct * percentage_bias
        ]
        
        gas_names = ['Cheap Item', 'Expensive Item']
        
        bars2 = ax2.bar(gas_names, perceived_values, color=['orange', 'purple'], alpha=0.7)
        ax2.axhline(10, color='green', linestyle='--', label='Actual Savings ($10)')
        ax2.set_ylabel('Perceived Value of Savings ($)')
        ax2.set_title('Percentage Bias in Savings')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"🧠 Mental Accounting Analysis:")
        print(f"• Lost ticket feels like ${theater_costs[0]:.0f} vs lost cash ${theater_costs[1]:.0f}")
        print(f"• $10 savings on cheap item feels worth ${perceived_values[0]:.1f}")
        print(f"• $10 savings on expensive item feels worth ${perceived_values[1]:.1f}")
        print(f"\n💡 Same $10 savings, but {perceived_values[0]/perceived_values[1]:.1f}x difference in perceived value!")

mental_accounting_demo()

## 4. Agent-Based Market Simulation

*"The best way to understand a system is to try to build it."* — Richard Feynman

Now let's see how these behavioral biases interact in a simulated market. We'll create agents with different behavioral characteristics and watch how markets evolve when populated by real humans rather than perfectly rational actors.

In [ ]:
def behavioral_market_simulation():
    @widgets.interact(
        n_agents=widgets.IntSlider(value=100, min=50, max=200, step=10, description='Number of Agents'),
        n_periods=widgets.IntSlider(value=50, min=20, max=100, step=5, description='Trading Periods'),
        behavioral_fraction=widgets.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.05, description='% Behavioral Agents'),
        volatility=widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.01, description='Fundamental Volatility')
    )
    def run_simulation(n_agents, n_periods, behavioral_fraction, volatility):
        np.random.seed(42)
        
        market = BehavioralMarket(
            n_agents=n_agents,
            behavioral_fraction=behavioral_fraction,
            fundamental_volatility=volatility
        )
        
        results = market.simulate(n_periods)
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
        
        periods = range(n_periods)
        
        ax1.plot(periods, results['prices'], 'b-', linewidth=2, label='Market Price')
        ax1.plot(periods, results['fundamentals'], 'g--', linewidth=2, label='Fundamental Value')
        ax1.fill_between(periods, results['prices'], results['fundamentals'], 
                        alpha=0.3, color='red', label='Mispricing')
        ax1.set_xlabel('Period')
        ax1.set_ylabel('Price')
        ax1.set_title('Price vs Fundamental Value')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        ax2.plot(periods, results['volumes'], 'purple', linewidth=2)
        ax2.set_xlabel('Period')
        ax2.set_ylabel('Trading Volume')
        ax2.set_title('Trading Volume Over Time')
        ax2.grid(True, alpha=0.3)
        
        mispricing = np.array(results['prices']) - np.array(results['fundamentals'])
        ax3.hist(mispricing, bins=20, alpha=0.7, color='orange', edgecolor='black')
        ax3.axvline(0, color='red', linestyle='--', linewidth=2, label='Perfect Efficiency')
        ax3.set_xlabel('Mispricing (Price - Fundamental)')
        ax3.set_ylabel('Frequency')
        ax3.set_title('Distribution of Mispricing')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        returns = np.diff(results['prices']) / results['prices'][:-1]
        ax4.plot(periods[1:], returns, 'red', alpha=0.7, linewidth=1)
        ax4.axhline(0, color='black', linestyle='-', alpha=0.5)
        ax4.set_xlabel('Period')
        ax4.set_ylabel('Return')
        ax4.set_title('Price Returns')
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        avg_mispricing = np.mean(np.abs(mispricing))
        volatility_ratio = np.std(returns) / volatility
        avg_volume = np.mean(results['volumes'])
        
        print(f"📊 Market Simulation Results:")
        print(f"• {n_agents} agents, {behavioral_fraction:.0%} behavioral")
        print(f"• Average absolute mispricing: {avg_mispricing:.3f}")
        print(f"• Price volatility / Fundamental volatility: {volatility_ratio:.2f}")
        print(f"• Average trading volume: {avg_volume:.1f}")
        
        if avg_mispricing > 0.05:
            print(f"\n💡 Significant market inefficiency detected!")
            print(f"Behavioral biases are creating persistent mispricing.")
        
        if volatility_ratio > 1.5:
            print(f"💡 Excess volatility detected!")
            print(f"Prices are more volatile than fundamentals justify.")

behavioral_market_simulation()

## Conclusion: The Beautiful Complexity of Human Decision-Making

*"I learned very early the difference between knowing the name of something and knowing something."* — Richard Feynman

We've journeyed through the landscape of behavioral economics, from the mathematical elegance of prospect theory to the messy reality of market simulations. What have we learned?


1. **Humans are predictably "irrational"**: Our deviations from rational choice theory aren't random errors — they're systematic patterns that can be modeled and predicted.

2. **Context matters enormously**: The same choice can elicit different responses depending on framing, reference points, and mental accounting categories.

3. **Biases interact in complex ways**: Loss aversion amplifies framing effects, anchoring influences mental accounting, and these interactions create rich behavioral patterns.

4. **Markets don't eliminate biases**: Even with arbitrageurs, behavioral biases can create persistent inefficiencies and excess volatility.


Feynman once said, "If you want to learn about nature, to appreciate nature, it is necessary to understand the language that she speaks in." The same is true for human behavior. Traditional economics spoke in the language of optimization and equilibrium. Behavioral economics speaks in the language of psychology and bounded rationality.

Neither language is "wrong" — they're useful for different purposes. Traditional theory gives us powerful tools for understanding market mechanisms and policy effects. Behavioral theory helps us understand why people make the choices they do and how to design better institutions.


The future of economics lies not in choosing between rational and behavioral approaches, but in understanding when each applies. Some decisions are made carefully with full information (buying a house). Others are made quickly with limited attention (choosing a snack). The art is knowing which model fits which situation.

As we continue to develop this field, remember Feynman's advice: "Study hard what interests you the most in the most undisciplined, irreverent and original manner possible." Human behavior is endlessly fascinating precisely because it's not perfectly predictable. That's not a bug — it's a feature.


Economics is ultimately about human flourishing. By understanding how people actually make decisions — with all their biases, heuristics, and systematic "errors" — we can design better policies, institutions, and choice environments. We can help people make decisions that align with their true preferences and long-term well-being.

That's the real power of behavioral economics: not just understanding human nature, but using that understanding to make the world a little bit better.

---

*"The worthwhile problems are the ones you can really solve or help solve, the ones you can really contribute something to."* — Richard Feynman

Welcome to the wonderful world of behavioral economics. Now go forth and solve some worthwhile problems.